[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/dirac_waves/dirac_waves.ipynb)

# Spinor Waves on a Triangulated Sphere

This example propagates a spinor wave pulse across a triangulated sphere and solves for the surface's standing spinor modes. A localized pulse started at the north pole spreads as a circular ring and focuses onto the south pole half a turn around the sphere later, while the standing waves oscillate at integer frequencies whose multiplicities grow as eight times the frequency.

Each vertex carries an even multivector spinor (`Even+[V]`), and the surface's sparse Dirac map `dirac` (`Point-[F <- V]`) sums the three corner spinors of each triangle weighted by their opposite intrinsic edge vectors over twice the triangle area, producing a triangle spinor `Even-[F]`. Reversing the sparse map, `~dirac`, reverses both the edge factors and the incidence direction so couplings run from triangles back to corners; alternating `dirac` and `~dirac` steps waves symplectically, while `(dirac_squared * Even).eigh(vertex_areas * Even, n_modes)` solves the standing-wave eigenproblem.

In the quaternionic matrix notation of Crane, Pinkall, and Schröder, each triangle corner coupling is assembled as an imaginary quaternion matrix entry $D_{ij} = -\tfrac{1}{2\mathcal A_i} e_j$ under Hamilton's sign convention, and standing modes solve the block generalized eigenproblem $D^* M_2 D\,\psi = \lambda M_0\,\psi$ over $4|V|$ real components with diagonal area matrices $M_2$ and $M_0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology
from examples.surfaces.dirac_waves import render, scenarios
from examples.util import shapes
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-
Even = ga.gatype.even()                             # Even+
SpinorPair = tuple[Even, Even]

## Sparse Dirac map on a triangulated sphere

Each triangle's boundary edges are intrinsic points (`Point-[F]`), and multiplying an intrinsic point into a vertex spinor (`Even+[V]`) produces an intrinsic triangle spinor (`Even-[F]`). Dividing each corner's opposite edge by twice the triangle area forms the sparse Dirac map `dirac`, while lumping one third of every incident triangle's area onto its corners forms the dual vertex areas.

In [ ]:
n_subdivisions = scenarios.n_subdivisions
bump_width = scenarios.bump_width

In [ ]:
def dirac_system(topology: Topology, vertices: Point) -> tuple[SparseExtensor, SparseExtensor, SparseExtensor]:
    """The triangulated surface (`Point-[V]`) defines the sparse Dirac map (`Point-[F <- V]`) alongside the diagonal triangle-area (`Scalar+[F <- F]`) and dual-area (`Scalar+[V <- V]`) maps."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                # [3] Point-[F]
    areas = sides[1].commutator(sides[0]).norm() / 2                            # Scalar+[F]
    corner_mean = topology.incidence[2, 0].mean(axis=0)                         # Scalar+[F <- V]
    dual_areas = corner_mean.adjugate() * areas                                 # Scalar+[V]
    face_areas = spdiag(areas)                                                  # Scalar+[F <- F]
    vertex_areas = spdiag(dual_areas)                                           # Scalar+[V <- V]
    dirac = (spdiag(sides / (2 * areas)) * topology.incidence[2, 0]).sum(axis=0)   # Point-[F <- V]
    return dirac, face_areas, vertex_areas


topology, coordinates = shapes.sphere(n_subdivisions)
vertices = mv.vector(coordinates).dual().field()                                # Point-[V]
dirac, face_areas, vertex_areas = dirac_system(topology, vertices)
initial_spinor = ((vertices - mv.z.dual()).norm_squared() * (-0.5 / bump_width**2)).exp() * mv.xy   # Even+[V]
save_figure(render.spinor_figure(topology, vertices, initial_spinor, dirac * initial_spinor), "dirac_waves_bump");

Red, green, and blue show the `yz`, `zx`, and `xy` bivector amplitudes of the vertex and triangle spinors combined: the blue `xy` Gaussian bump sits at the north pole, surrounded by red and green `yz` and `zx` ring components generated by `dirac * initial_spinor`.

In [ ]:
print(dirac.gatype, dirac.shape)
print((~dirac).gatype, (~dirac).shape)

## Antipodal focusing of a spinor pulse

Advancing the triangle spinor by `dirac * vertex_spinor` and pulling the result back to the vertices through `~dirac * (face_areas * face_spinor)` divided by the dual vertex areas forms a leapfrog integrator that preserves discrete wave energy. Over a unit sphere, a pulse launched at the north pole expands across the equator and reconverges at the south pole after time `pi`.

In [ ]:
dt = scenarios.dt
n_steps = scenarios.n_steps
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def pulse_states(
    dirac: SparseExtensor,
    face_areas: SparseExtensor,
    vertex_areas: SparseExtensor,
    initial_spinor: Even,
    dt: float,
    n_steps: int,
) -> Iterator[SpinorPair]:
    """Yields `(vertex_spinor, face_spinor)` (`Even+[V]`, `Even-[F]`) across `n_steps` leapfrog steps of step size `dt`."""
    inverse_vertex_areas = spdiag(1 / vertex_areas.diagonal())                  # Scalar+[V <- V]
    vertex_spinor = initial_spinor                                              # Even+[V]
    face_spinor = dirac * initial_spinor * 0.0                                  # Even-[F]
    for _ in range(n_steps):
        # Advance the staggered face spinor by half a step under the Dirac map.
        face_spinor = face_spinor + dirac * vertex_spinor * dt                  # Even-[F]
        yield vertex_spinor, face_spinor
        # Pull the face spinor back to the vertices through the area-weighted reverse Dirac map.
        vertex_spinor = vertex_spinor - inverse_vertex_areas * (~dirac * (face_areas * face_spinor)) * dt   # Even+[V]


recorded_pulse = show_states(
    pulse_states(dirac, face_areas, vertex_areas, initial_spinor, dt, n_steps),
    render.pulse_animation,
    topology,
    vertices,
    name="dirac_waves_pulse",
    duration_ms=frame_duration_ms,
    n_frames=n_steps,
);

As the blue north-polar bump spreads outward, the Dirac coupling mixes `yz` and `zx` planes around the advancing wavefront before gathering the wave packet at the south pole.

## Standing spinor modes on the sphere

Composing `dirac` with its area-weighted reverse gives the symmetric vertex operator `dirac_squared = ~dirac * face_areas * dirac`. Multiplying by `Even` specifies the spinor domain `(Even+ <- Even+)[V <- V]` for the generalized eigensolver against `vertex_areas * Even`; projecting each degenerate shell onto an equatorial bump selects one aligned standing mode per frequency shell.

In [ ]:
n_levels = scenarios.n_levels
n_constant = scenarios.n_constant
n_phases = scenarios.n_phases

In [ ]:
def standing_modes(
    dirac: SparseExtensor,
    face_areas: SparseExtensor,
    vertex_areas: SparseExtensor,
    level_bounds: np.ndarray,
    probe: Even,
    frequencies: np.ndarray,
) -> tuple[Scalar, Even, Even]:
    """Eigenfields of `~dirac * face_areas * dirac` against `vertex_areas` determine the squared frequencies (`[n_modes] Scalar+`) and one aligned `(vertex_modes, face_modes)` pair per shell."""
    n_modes = int(level_bounds[-1])
    dirac_squared = ~dirac * face_areas * dirac                                 # Even+[V <- V]
    squared_frequencies, eigenfields = (dirac_squared * Even).eigh(vertex_areas * Even, n_modes)   # [n_modes] Scalar+, [n_modes] Even+[V]
    shells = [eigenfields[low:high] for low, high in zip(level_bounds[:-1], level_bounds[1:])]
    vertex_modes = stack([
        (shell * (vertex_areas * shell.reverse().scalar_product(probe)).sites.sum()).sum(axis=0)
        for shell in shells
    ])                                                                          # [...] Even+[V]
    face_modes = dirac * vertex_modes * (1 / frequencies)                       # [...] Even-[F]
    return squared_frequencies, vertex_modes, face_modes


def mode_states(vertex_modes: Even, face_modes: Even, phases: np.ndarray) -> Iterator[SpinorPair]:
    """Yields `(vertex_modes * cos(phase), face_modes * sin(phase))` across each oscillation `phase` in `phases` (`[n_phases]`)."""
    for phase in phases:
        yield vertex_modes * float(np.cos(phase)), face_modes * float(np.sin(phase))


level_bounds = n_constant + 8 * np.cumsum(np.arange(n_levels + 1))              # [n_levels + 1]
probe = ((vertices - mv.x.dual()).norm_squared() * (-0.5 / bump_width**2)).exp() * (mv.yz + mv.zx + mv.xy)   # Even+[V]
frequencies = np.arange(1, n_levels + 1)                                        # [n_levels]
phases = np.linspace(0.0, 2 * np.pi, n_phases, endpoint=False)                  # [n_phases]
squared_frequencies, vertex_modes, face_modes = standing_modes(
    dirac, face_areas, vertex_areas, level_bounds, probe, frequencies,
)
recorded_modes = show_states(
    mode_states(vertex_modes, face_modes, phases),
    render.modes_animation,
    topology,
    vertices,
    n_levels,
    name="dirac_waves_modes",
    duration_ms=frame_duration_ms,
    n_frames=n_phases,
);

From left to right, the five nonzero spherical Dirac shells add nodal rings around the equatorial bump while exchanging amplitude between vertex spinors and triangle spinors every quarter period.

In [ ]:
# checks
sample_face = recorded_modes[n_phases // 4][1][0]                               # Even-[F]
forward_pair = (face_areas * sample_face.reverse().scalar_product(dirac * probe)).sites.sum()   # Scalar+
reverse_pair = (~dirac * (face_areas * sample_face)).reverse().scalar_product(probe).sites.sum()   # Scalar+
assert (forward_pair - reverse_pair).abs() < 1e-12 * forward_pair.abs()

energies = stack([
    (vertex_areas * v_after.scalar_norm_squared()).sites.sum()
    + (face_areas * f_before.reverse().scalar_product(f_after)).sites.sum()
    for (v_after, f_after), (_, f_before) in zip(recorded_pulse[1:], recorded_pulse)
])                                                                              # [n_steps - 1] Scalar+
assert ((energies - energies[0]).abs() < 1e-12 * energies[0].abs()).all()

antipodal_spinor = recorded_pulse[int(round(np.pi / dt))][0]                    # Even+[V]
vertex_energy = (vertex_areas * antipodal_spinor.scalar_norm_squared()).batch() # [V] Scalar+
south_cap = (mv.z & vertices.batch()) < -0.9
assert bool(vertex_energy[south_cap].sum() > 0.5 * vertex_energy.sum())

assert (squared_frequencies[:n_constant].abs() < 1e-9).all()
expected = np.repeat(frequencies, 8 * frequencies)
assert (((squared_frequencies[n_constant:].square_root() - expected) / expected).abs() < 0.03).all()